# 07_rag_qa_system

Notebook นี้เป็นต้นแบบระบบถามตอบจากความรู้ทั้งหมดในโปรเจกต์ Welly AI  
แนวคิดคือรวม knowledge จากหลายแหล่ง แล้วใช้ retrieval แบบ TF-IDF เพื่อดึงข้อมูลที่เกี่ยวข้องกับคำถามของผู้ใช้

## เป้าหมาย
- รวมความรู้จาก nutrition standards, DGA, BMI, user health, และ food risk
- สร้าง knowledge base กลาง
- ทำระบบถามตอบแบบ retrieval-based คล้าย RAG เบื้องต้น
- ทดสอบถามคำถามจากข้อมูลจริงในโปรเจกต์


## 1. Import Libraries

In [1]:
import os
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)


## 2. Check Required Files

Notebook นี้รันจากโฟลเดอร์ `Notebooks/`  
ดังนั้น path หลักจะอิง `../data/` และ `../outputs/`


In [2]:
knowledge_files = {
    "standard": "../data/knowledge/standard_df.csv",
    "dga_standard": "../data/knowledge/dga_standard_df.csv",
    "bmi_standard": "../data/knowledge/bmi_standard_df.csv",
    "bmi_rules": "../data/knowledge/bmi_rules_df.csv",
    "user_health": "../data/knowledge/user_health_knowledge.csv",
    "food_risk": "../outputs/food_dataset_with_risk.csv"
}

for name, path in knowledge_files.items():
    print(f"{name:15s} -> {path} -> {os.path.exists(path)}")


standard        -> ../data/knowledge/standard_df.csv -> True
dga_standard    -> ../data/knowledge/dga_standard_df.csv -> True
bmi_standard    -> ../data/knowledge/bmi_standard_df.csv -> True
bmi_rules       -> ../data/knowledge/bmi_rules_df.csv -> True
user_health     -> ../data/knowledge/user_health_knowledge.csv -> True
food_risk       -> ../outputs/food_dataset_with_risk.csv -> True


## 3. Load Knowledge Tables

In [3]:
missing = [name for name, path in knowledge_files.items() if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(f"ยังไม่พบไฟล์เหล่านี้: {missing}")

standard_df = pd.read_csv(knowledge_files["standard"])
dga_standard_df = pd.read_csv(knowledge_files["dga_standard"])
bmi_standard_df = pd.read_csv(knowledge_files["bmi_standard"])
bmi_rules_df = pd.read_csv(knowledge_files["bmi_rules"])
user_health_df = pd.read_csv(knowledge_files["user_health"])
food_risk_df = pd.read_csv(knowledge_files["food_risk"])

print("standard_df:", standard_df.shape)
print("dga_standard_df:", dga_standard_df.shape)
print("bmi_standard_df:", bmi_standard_df.shape)
print("bmi_rules_df:", bmi_rules_df.shape)
print("user_health_df:", user_health_df.shape)
print("food_risk_df:", food_risk_df.shape)


standard_df: (7, 6)
dga_standard_df: (8, 7)
bmi_standard_df: (10, 9)
bmi_rules_df: (3, 5)
user_health_df: (5000, 17)
food_risk_df: (2395, 20)


## 4. Preview Tables

In [4]:
display(standard_df.head())
display(dga_standard_df.head())
display(bmi_standard_df.head())
display(user_health_df.head())
display(food_risk_df.head())


,source_doc,category,metric,recommended_value,unit,note
0,media.pdf,daily_reference,Energy,2000,kcal/day,Thai RDI reference base
1,media.pdf,daily_reference,Total Fat,65,g/day,Thai RDI reference
2,media.pdf,daily_reference,Saturated Fat,20,g/day,Thai RDI reference
3,media.pdf,daily_reference,Cholesterol,300,mg/day,Thai RDI reference
4,media.pdf,daily_reference,Total Carbohydrate,300,g/day,Thai RDI reference


,source_doc,category,metric,min_value,max_value,unit,target_group
0,DGA.pdf,Protein,protein_intake,1.2,1.6,g/kg/day,general
1,DGA.pdf,Dairy,dairy_servings,3.0,3.0,servings/day,2000_kcal_pattern
2,DGA.pdf,Vegetables,vegetable_servings,3.0,3.0,servings/day,2000_kcal_pattern
3,DGA.pdf,Fruits,fruit_servings,2.0,2.0,servings/day,2000_kcal_pattern
4,DGA.pdf,Whole Grains,whole_grain_servings,2.0,4.0,servings/day,general


,source_doc,category,metric,min_value,max_value,unit,target_group,label,note
0,document-20210831192536.pdf,BMI,bmi_formula,NaN,NaN,kg/m^2,general,BMI = weight_kg / (height_m ** 2),ใช้สูตรน้ำหนัก(กก.) / ส่วนสูง(เมตร)^2
1,document-20210831192536.pdf,BMI,bmi_category,-inf,18.49,kg/m^2,asian_adults,Underweight,น้ำหนักต่ำกว่าเกณฑ์
2,document-20210831192536.pdf,BMI,bmi_category,18.5,22.99,kg/m^2,asian_adults,Normal,น้ำหนักปกติ
3,document-20210831192536.pdf,BMI,bmi_category,23.0,24.99,kg/m^2,asian_adults,Overweight,น้ำหนักเกิน
4,document-20210831192536.pdf,BMI,bmi_category,25.0,29.99,kg/m^2,asian_adults,Obese Level 1,อ้วนระดับ 1


,Patient_ID,Age,Gender,Height_cm,Weight_kg,BMI,BMI_Category,Blood_Pressure_Systolic,Blood_Pressure_Diastolic,Blood_Sugar_Level,Cholesterol_Level,Health_Profile_Summary,Recommended_Calories,Recommended_Protein,Recommended_Carbs,Recommended_Fats,Recommended_Meal_Plan
0,P00001,56,Other,163,66,24.84,Overweight,175,75,124,219,Overall moderate cardiometabolic risk. BMI: Ov...,2150,108,139,145,High-Protein Diet
1,P00002,69,Female,171,114,38.99,Obese Level 2,155,72,72,208,Overall moderate cardiometabolic risk. BMI: Ob...,1527,74,266,80,Balanced Diet
2,P00003,46,Female,172,119,40.22,Obese Level 2,137,101,145,171,Overall high cardiometabolic risk. BMI: Obese ...,2359,180,145,143,High-Protein Diet
3,P00004,32,Female,197,118,30.41,Obese Level 2,148,91,235,258,Overall high cardiometabolic risk. BMI: Obese ...,2858,137,378,135,High-Protein Diet
4,P00005,60,Female,156,109,44.79,Obese Level 2,160,109,248,260,Overall high cardiometabolic risk. BMI: Obese ...,1937,166,317,56,High-Protein Diet


,food_name,calories,fat,sat_fat,carbs,sugar,protein,fiber,cholesterol,sodium,sodium_pct_daily,cholesterol_pct_daily,sat_fat_pct_daily,fat_pct_daily,carbs_pct_daily,fiber_pct_daily,sugar_pct_meal_limit,risk_level,risk_label,chatbot_summary
0,cream cheese,51,5.0,2.9,0.8,0.500,0.9,0.0,14.6,0.016,0.00080,4.866667,14.5,7.692308,0.266667,0.0,5.00,Low,0,ยังไม่พบตัวชี้วัดที่เกินเกณฑ์เบื้องต้น
1,neufchatel cheese,215,19.4,10.9,3.1,2.700,7.8,0.0,62.9,0.300,0.01500,20.966667,54.5,29.846154,1.033333,0.0,27.00,Medium,1,โคเลสเตอรอลค่อนข้างสูง | ไขมันอิ่มตัวค่อนข้างสูง
2,requeijao cremoso light catupiry,49,3.6,2.3,0.9,3.400,0.8,0.1,0.0,0.000,0.00000,0.000000,11.5,5.538462,0.300000,0.4,34.00,Low,0,ยังไม่พบตัวชี้วัดที่เกินเกณฑ์เบื้องต้น
3,ricotta cheese,30,2.0,1.3,1.5,0.091,1.5,0.0,9.8,0.017,0.00085,3.266667,6.5,3.076923,0.500000,0.0,0.91,Low,0,ยังไม่พบตัวชี้วัดที่เกินเกณฑ์เบื้องต้น
4,cream cheese low fat,30,2.3,1.4,1.2,0.900,1.2,0.0,8.1,0.046,0.00230,2.700000,7.0,3.538462,0.400000,0.0,9.00,Low,0,ยังไม่พบตัวชี้วัดที่เกินเกณฑ์เบื้องต้น


## 5. Convert Tables to Text Knowledge

ในส่วนนี้จะเปลี่ยนแต่ละ row ให้กลายเป็นข้อความ 1 ชิ้น  
เพื่อให้ระบบ retrieval ค้นหาได้


In [5]:
knowledge_rows = []

def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [6]:
for _, row in standard_df.iterrows():
    text = f'''
    Nutrition standard metric {row.get("metric", "")}.
    Category {row.get("category", "")}.
    Recommended value {row.get("recommended_value", "")} {row.get("unit", "")}.
    Note {row.get("note", "")}.
    '''
    knowledge_rows.append({
        "source": "standard_df",
        "text": clean_text(text)
    })

for _, row in dga_standard_df.iterrows():
    text = f'''
    DGA standard metric {row.get("metric", "")}.
    Category {row.get("category", "")}.
    Minimum value {row.get("min_value", "")}.
    Maximum value {row.get("max_value", "")}.
    Unit {row.get("unit", "")}.
    Target group {row.get("target_group", "")}.
    '''
    knowledge_rows.append({
        "source": "dga_standard_df",
        "text": clean_text(text)
    })

for _, row in bmi_standard_df.iterrows():
    text = f'''
    BMI or body measurement standard metric {row.get("metric", "")}.
    Category {row.get("category", "")}.
    Label {row.get("label", "")}.
    Minimum value {row.get("min_value", "")}.
    Maximum value {row.get("max_value", "")}.
    Unit {row.get("unit", "")}.
    Target group {row.get("target_group", "")}.
    Note {row.get("note", "")}.
    '''
    knowledge_rows.append({
        "source": "bmi_standard_df",
        "text": clean_text(text)
    })

for _, row in bmi_rules_df.iterrows():
    text = f'''
    BMI rule {row.get("rule_name", "")}.
    Condition {row.get("condition", "")}.
    Unit {row.get("unit", "")}.
    Note {row.get("note", "")}.
    '''
    knowledge_rows.append({
        "source": "bmi_rules_df",
        "text": clean_text(text)
    })


In [7]:
# User health knowledge
candidate_patient_col = None
for c in ["Patient_ID", "patient_id", "User_ID", "user_id"]:
    if c in user_health_df.columns:
        candidate_patient_col = c
        break

for _, row in user_health_df.iterrows():
    text = f'''
    User health profile.
    Patient id {row.get(candidate_patient_col, "")}.
    Age {row.get("Age", "")}.
    Gender {row.get("Gender", "")}.
    Height {row.get("Height_cm", "")} cm.
    Weight {row.get("Weight_kg", "")} kg.
    BMI {row.get("BMI", "")}.
    BMI Category {row.get("BMI_Category", "")}.
    Blood Pressure Systolic {row.get("Blood_Pressure_Systolic", "")}.
    Blood Pressure Diastolic {row.get("Blood_Pressure_Diastolic", "")}.
    Blood Sugar Level {row.get("Blood_Sugar_Level", "")}.
    Cholesterol Level {row.get("Cholesterol_Level", "")}.
    Summary {row.get("Health_Profile_Summary", "")}.
    Recommended Calories {row.get("Recommended_Calories", "")}.
    Recommended Protein {row.get("Recommended_Protein", "")}.
    Recommended Carbs {row.get("Recommended_Carbs", "")}.
    Recommended Fats {row.get("Recommended_Fats", "")}.
    Recommended Meal Plan {row.get("Recommended_Meal_Plan", "")}.
    '''
    knowledge_rows.append({
        "source": "user_health_knowledge",
        "text": clean_text(text)
    })


In [8]:
# Food risk knowledge
food_name_col = None
for c in ["food_name", "Food", "food", "Food Name"]:
    if c in food_risk_df.columns:
        food_name_col = c
        break

for _, row in food_risk_df.iterrows():
    text = f'''
    Food profile.
    Food name {row.get(food_name_col, "") if food_name_col else ""}.
    Calories {row.get("calories", "")}.
    Fat {row.get("fat", "")}.
    Saturated fat {row.get("sat_fat", "")}.
    Sugar {row.get("sugar", "")}.
    Protein {row.get("protein", "")}.
    Fiber {row.get("fiber", "")}.
    Cholesterol {row.get("cholesterol", "")}.
    Sodium {row.get("sodium", "")}.
    Risk level {row.get("risk_level", "")}.
    Chatbot summary {row.get("chatbot_summary", "")}.
    '''
    knowledge_rows.append({
        "source": "food_dataset_with_risk",
        "text": clean_text(text)
    })


In [9]:
knowledge_base = pd.DataFrame(knowledge_rows)
print("knowledge_base shape:", knowledge_base.shape)
knowledge_base.head()


knowledge_base shape: (7423, 2)


,source,text
0,standard_df,Nutrition standard metric Energy. Category dai...
1,standard_df,Nutrition standard metric Total Fat. Category ...
2,standard_df,Nutrition standard metric Saturated Fat. Categ...
3,standard_df,Nutrition standard metric Cholesterol. Categor...
4,standard_df,Nutrition standard metric Total Carbohydrate. ...


## 6. Build Retriever with TF-IDF

In [10]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
X_knowledge = vectorizer.fit_transform(knowledge_base["text"])

print("Vector shape:", X_knowledge.shape)


Vector shape: (7423, 43990)


In [11]:
def retrieve_knowledge(query, top_k=5):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, X_knowledge).flatten()
    top_indices = scores.argsort()[::-1][:top_k]

    results = knowledge_base.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    return results[["source", "text", "score"]].reset_index(drop=True)


## 7. Simple Answer Generator

ส่วนนี้ยังไม่ใช้ LLM จริง  
แต่จะสรุปจาก knowledge ที่ค้นเจอให้เป็นคำตอบเบื้องต้น


In [12]:
def generate_answer_from_results(query, retrieved_df):
    lines = []
    lines.append(f"Question: {query}")
    lines.append("")
    lines.append("Answer (retrieval-based):")

    if retrieved_df.empty:
        lines.append("No related knowledge found.")
        return "\n".join(lines)

    for i, row in retrieved_df.iterrows():
        lines.append(f"{i+1}. [{row['source']}] {row['text']}")

    return "\n".join(lines)

def answer_question(query, top_k=5):
    retrieved = retrieve_knowledge(query, top_k=top_k)
    return generate_answer_from_results(query, retrieved)


## 8. Test Questions

In [13]:
print(answer_question("What is the normal BMI range for Asian adults?", top_k=3))


Question: What is the normal BMI range for Asian adults?

Answer (retrieval-based):
1. [food_dataset_with_risk] Food profile. Food name asian pear. Calories 51. Fat 0.3. Saturated fat 0.009. Sugar 8.6. Protein 0.6. Fiber 4.4. Cholesterol 0.0. Sodium 0.0. Risk level Low. Chatbot summary ยังไม่พบตัวชี้วัดที่เกินเกณฑ์เบื้องต้น.
2. [user_health_knowledge] User health profile. Patient id P03067. Age 27. Gender Female. Height 162 cm. Weight 53 kg. BMI 20.2. BMI Category Normal. Blood Pressure Systolic 119. Blood Pressure Diastolic 78. Blood Sugar Level 84. Cholesterol Level 154. Summary Overall lower cardiometabolic risk. BMI: Normal (20.20). Blood pressure: Normal (119/78 mmHg). Blood sugar: Normal (84 mg/dL). Cholesterol: Desirable (154 mg/dL). Daily activity: Low activity (2,729 steps/day).. Recommended Calories 2675. Recommended Protein 103. Recommended Carbs 144. Recommended Fats 35. Recommended Meal Plan High-Protein Diet.
3. [user_health_knowledge] User health profile. Patient id P017

In [45]:
print(answer_question("How much sodium per day is recommended?", top_k=3))


Question: How much sodium per day is recommended?

Answer (retrieval-based):
1. [user_health_knowledge] User health profile. Patient id P00027. Age 55. Gender Male. Height 168 cm. Weight 59 kg. BMI 20.9. BMI Category Normal. Blood Pressure Systolic 167. Blood Pressure Diastolic 96. Blood Sugar Level 159. Cholesterol Level 193. Summary Overall moderate cardiometabolic risk. BMI: Normal (20.90). Blood pressure: Stage 2 hypertension range (167/96 mmHg). Blood sugar: High (159 mg/dL). Cholesterol: Desirable (193 mg/dL). Daily activity: Active (13,410 steps/day).. Recommended Calories 2781. Recommended Protein 100. Recommended Carbs 219. Recommended Fats 73. Recommended Meal Plan Low-Fat Diet.
2. [user_health_knowledge] User health profile. Patient id P01207. Age 41. Gender Male. Height 168 cm. Weight 92 kg. BMI 32.6. BMI Category Obese Level 2. Blood Pressure Systolic 117. Blood Pressure Diastolic 100. Blood Sugar Level 114. Cholesterol Level 164. Summary Overall high cardiometabolic risk.

In [43]:
print(answer_question("What is the waist risk threshold for female users?", top_k=3))


Question: What is the waist risk threshold for female users?

Answer (retrieval-based):
1. [bmi_standard_df] BMI or body measurement standard metric waist_risk_threshold. Category Waist Circumference. Label Normal. Minimum value nan. Maximum value 80.0. Unit cm. Target group female. Note ผู้หญิงควรมีเส้นรอบเอวไม่เกิน 80 ซม..
2. [bmi_standard_df] BMI or body measurement standard metric waist_risk_threshold. Category Waist Circumference. Label High Risk. Minimum value 80.01. Maximum value inf. Unit cm. Target group female. Note ผู้หญิงเสี่ยงเมื่อเส้นรอบเอวเกิน 80 ซม..
3. [bmi_standard_df] BMI or body measurement standard metric waist_risk_threshold. Category Waist Circumference. Label Normal. Minimum value nan. Maximum value 90.0. Unit cm. Target group male. Note ผู้ชายควรมีเส้นรอบเอวไม่เกิน 90 ซม..


In [16]:
print(answer_question("Which foods are high risk because of sodium or cholesterol?", top_k=5))


Question: Which foods are high risk because of sodium or cholesterol?

Answer (retrieval-based):
1. [food_dataset_with_risk] Food profile. Food name chow mein stir fry master foods. Calories 26. Fat 0.2. Saturated fat 5.5. Sugar 0.0. Protein 0.0. Fiber 0.0. Cholesterol 0.0. Sodium 0.0. Risk level Low. Chatbot summary ยังไม่พบตัวชี้วัดที่เกินเกณฑ์เบื้องต้น.
2. [bmi_standard_df] BMI or body measurement standard metric waist_risk_threshold. Category Waist Circumference. Label High Risk. Minimum value 90.01. Maximum value inf. Unit cm. Target group male. Note ผู้ชายเสี่ยงเมื่อเส้นรอบเอวเกิน 90 ซม..
3. [bmi_standard_df] BMI or body measurement standard metric waist_risk_threshold. Category Waist Circumference. Label High Risk. Minimum value 80.01. Maximum value inf. Unit cm. Target group female. Note ผู้หญิงเสี่ยงเมื่อเส้นรอบเอวเกิน 80 ซม..
4. [food_dataset_with_risk] Food profile. Food name salt. Calories 0. Fat 0.0. Saturated fat 0.0. Sugar 0.0. Protein 0.0. Fiber 0.0. Cholesterol 0.0. So

In [17]:
print(answer_question("Which users have high blood sugar and what meal plan is recommended?", top_k=5))


Question: Which users have high blood sugar and what meal plan is recommended?

Answer (retrieval-based):
1. [user_health_knowledge] User health profile. Patient id P01788. Age 26. Gender Other. Height 193 cm. Weight 111 kg. BMI 29.8. BMI Category Obese Level 1. Blood Pressure Systolic 145. Blood Pressure Diastolic 86. Blood Sugar Level 177. Cholesterol Level 276. Summary Overall high cardiometabolic risk. BMI: Obese Level 1 (29.80). Blood pressure: Stage 2 hypertension range (145/86 mmHg). Blood sugar: High (177 mg/dL). Cholesterol: High (276 mg/dL). Daily activity: Low activity (3,157 steps/day).. Recommended Calories 1088. Recommended Protein 123. Recommended Carbs 138. Recommended Fats 118. Recommended Meal Plan High-Protein Diet.
2. [user_health_knowledge] User health profile. Patient id P00935. Age 33. Gender Male. Height 185 cm. Weight 90 kg. BMI 26.3. BMI Category Obese Level 1. Blood Pressure Systolic 161. Blood Pressure Diastolic 86. Blood Sugar Level 153. Cholesterol Level 2

## 9. Interactive Query

In [41]:
user_query = "What is the recommended protein intake?"
print(answer_question(user_query, top_k=5))


Question: What is the recommended protein intake?

Answer (retrieval-based):
1. [user_health_knowledge] User health profile. Patient id P01690. Age 49. Gender Male. Height 179 cm. Weight 99 kg. BMI 30.9. BMI Category Obese Level 2. Blood Pressure Systolic 120. Blood Pressure Diastolic 106. Blood Sugar Level 163. Cholesterol Level 162. Summary Overall high cardiometabolic risk. BMI: Obese Level 2 (30.90). Blood pressure: Stage 2 hypertension range (120/106 mmHg). Blood sugar: High (163 mg/dL). Cholesterol: Desirable (162 mg/dL). Daily activity: Lightly active (6,674 steps/day).. Recommended Calories 2269. Recommended Protein 113. Recommended Carbs 206. Recommended Fats 39. Recommended Meal Plan High-Protein Diet.
2. [user_health_knowledge] User health profile. Patient id P02073. Age 79. Gender Male. Height 156 cm. Weight 55 kg. BMI 22.6. BMI Category Normal. Blood Pressure Systolic 179. Blood Pressure Diastolic 77. Blood Sugar Level 192. Cholesterol Level 167. Summary Overall high cardi

## 10. Chatbot-style Function

ฟังก์ชันนี้ทำให้คำตอบดูเป็นข้อความตอบกลับมากขึ้น


In [19]:
def simple_chatbot(query, top_k=3):
    retrieved = retrieve_knowledge(query, top_k=top_k)

    if retrieved.empty:
        return "ไม่พบข้อมูลที่เกี่ยวข้อง"

    response = "จากข้อมูลที่ค้นเจอ:\n"
    for _, row in retrieved.iterrows():
        response += f"- [{row['source']}] {row['text']}\n"
    return response


In [20]:
print(simple_chatbot("What meal plan is recommended for users with high BMI?", top_k=4))


จากข้อมูลที่ค้นเจอ:
- [user_health_knowledge] User health profile. Patient id P01788. Age 26. Gender Other. Height 193 cm. Weight 111 kg. BMI 29.8. BMI Category Obese Level 1. Blood Pressure Systolic 145. Blood Pressure Diastolic 86. Blood Sugar Level 177. Cholesterol Level 276. Summary Overall high cardiometabolic risk. BMI: Obese Level 1 (29.80). Blood pressure: Stage 2 hypertension range (145/86 mmHg). Blood sugar: High (177 mg/dL). Cholesterol: High (276 mg/dL). Daily activity: Low activity (3,157 steps/day).. Recommended Calories 1088. Recommended Protein 123. Recommended Carbs 138. Recommended Fats 118. Recommended Meal Plan High-Protein Diet.
- [user_health_knowledge] User health profile. Patient id P00935. Age 33. Gender Male. Height 185 cm. Weight 90 kg. BMI 26.3. BMI Category Obese Level 1. Blood Pressure Systolic 161. Blood Pressure Diastolic 86. Blood Sugar Level 153. Cholesterol Level 248. Summary Overall high cardiometabolic risk. BMI: Obese Level 1 (26.30). Blood pressur

## 11. Save Full Knowledge Base

In [21]:
os.makedirs("../outputs", exist_ok=True)
knowledge_base.to_csv("../outputs/full_knowledge_base.csv", index=False)
print("saved ../outputs/full_knowledge_base.csv")


saved ../outputs/full_knowledge_base.csv


## 12. Summary

Notebook นี้เป็นต้นแบบระบบถามตอบเชิงความรู้ โดยใช้ retrieval แบบ TF-IDF  
ซึ่งสามารถดึงข้อมูลจากหลายแหล่งในโปรเจกต์มารวมกันได้ เช่น

- nutrition standard
- DGA
- BMI standard
- user health knowledge
- food risk results

ในอนาคตสามารถต่อยอดไปสู่ระบบ RAG ที่ใช้ embedding หรือ LLM ได้
